In [6]:
import os
import json
from typing import List, Dict, Any

from google import genai
from google.genai import types

# Read API key from environment variable
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Please set the GEMINI_API_KEY environment variable before running this notebook.")

# Initialize Gemini client
client = genai.Client(api_key=API_KEY)


In [7]:
# 2. Define Google Search tool, system prompt, and configs

# Google Search tool
search_tool = types.Tool(google_search={})

# System prompt: defines the Drug Information Agent and when to use search
DRUG_AGENT_SYSTEM_PROMPT = """
You are a Drug Information Extraction Agent for healthcare professionals and data scientists.

Your tasks:
1. Focus ONLY on drug information: indications, mechanisms of action, dosage and administration, routes,
   common and serious adverse events, warnings/contraindications, major drug-drug interactions,
   and key clinical trial highlights.
2. Always prioritize reliable sources: FDA labels, EMA, official prescribing information, major guidelines,
   or reputable medical references.
3. TOOL USAGE POLICY:
   - You have access to a google_search tool.
   - ALWAYS use google_search when:
       * The user asks for very recent information (e.g., 2023 or later label changes, new approvals,
         new safety warnings, "latest" updates, "as of now").
       * The drug is unfamiliar or rarely used.
   - You MAY answer from your internal knowledge without google_search only when:
       * The question is about well-established, stable information (classic indications,
         well-known common adverse events).
   - If you are not confident, call google_search instead of guessing.
4. You are NOT giving personal medical advice and must not make treatment decisions
   for individual patients.
5. Keep answers concise, structured, and explicitly mention uncertainties when they exist.
"""

# Config for free-form natural language / conversational answers
agent_config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=DRUG_AGENT_SYSTEM_PROMPT,
)

# Config for JSON-only responses (used for structured extraction)
json_agent_config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=DRUG_AGENT_SYSTEM_PROMPT,
    response_mime_type="application/json",
)


In [8]:
# 3. Stateless single-turn Drug Information Agent

def drug_agent(question: str) -> str:
    """
    Single-turn drug information question answering without conversation state.
    Uses gemini-2.0-flash + google_search (via agent_config).
    """
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[question],
        config=agent_config,
    )
    return response.text


In [9]:
# 4. Conversational Drug Information Agent with memory

# Global conversation history
conversation_history: List[types.Content] = []

def reset_conversation() -> None:
    """Clear the global conversation history."""
    conversation_history.clear()

def chat_drug_agent(user_message: str) -> str:
    """
    Multi-turn conversational Drug Information Agent.

    - Maintains conversation_history.
    - Decides when to use google_search according to the system prompt.
    """
    # 1) Append this user turn to the history
    conversation_history.append(
        types.Content(role="user", parts=[types.Part(text=user_message)])
    )

    # 2) Call the model with the full history
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=conversation_history,
        config=agent_config,
    )

    answer = response.text

    # 3) Append the model answer to the history
    conversation_history.append(
        types.Content(role="model", parts=[types.Part(text=answer)])
    )

    return answer


In [10]:
# 5. JSON-structured drug information extraction agent 

def drug_agent_json(question: str) -> Dict[str, Any]:
    """
    JSON-structured extraction variant.

    Expected JSON fields:
      - generic_name (str)
      - brand_names (list[str])
      - indications (list[str])
      - mechanism (str)
      - dosage_and_administration (str)
      - common_adverse_events (list[str])
      - serious_adverse_events (list[str])
      - boxed_warnings (list[str])
      - references (list[str])
    """
    json_instruction = (
        "Extract detailed drug information and respond ONLY as a JSON object with the "
        "following keys: generic_name, brand_names, indications, mechanism, "
        "dosage_and_administration, common_adverse_events, serious_adverse_events, "
        "boxed_warnings, references. Do NOT include any extra text outside the JSON object."
    )

    full_query = f"{json_instruction}\n\nUser question: {question}"

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[full_query],
        config=json_agent_config,
    )

    text = response.text
    # For debugging, you can uncomment:
    # print("RAW MODEL OUTPUT:\n", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback: return raw output for manual inspection
        return {"_parse_error": "JSONDecodeError", "_raw_text": text}


In [11]:
# 5A. Quick tests

print("=== Stateless drug_agent test ===")
print(drug_agent("What is Keytruda used for? List main indications and common adverse events."))

print("\n=== Conversational chat_drug_agent test ===")
reset_conversation()
print("Agent:", chat_drug_agent("Hi, I mainly treat lung cancer patients. Remember this."))
print("Agent:", chat_drug_agent("Based on my practice, what Keytruda indications are most relevant?"))
print("Agent:", chat_drug_agent("What immune-related adverse events should I watch for with Keytruda?"))

print("\n=== JSON drug_agent_json test ===")
info = drug_agent_json("Extract detailed drug information for Keytruda.")
info


=== Stateless drug_agent test ===
Keytruda (pembrolizumab) is a prescription medicine used to treat multiple cancers.

**Main Indications:**

*   **Melanoma:** Used for unresectable or metastatic melanoma and adjuvant treatment of adult and pediatric patients (12 years and older).
*   **Non-Small Cell Lung Cancer (NSCLC):** Used for resectable tumors (≥4 cm or node positive) in combination with platinum-containing chemotherapy as neoadjuvant treatment, and then continued as a single agent as adjuvant treatment after surgery. Also used as a single agent as adjuvant treatment following resection and platinum-based chemotherapy for adult patients with stage IB (T2a ≥4 cm), II, or IIIA NSCLC.
*   **Head and Neck Squamous Cell Carcinoma (HNSCC):** Used for recurrent or metastatic HNSCC with disease progression on or after platinum-containing chemotherapy. In combination with platinum and fluorouracil (FU) for first-line treatment of metastatic or unresectable, recurrent HNSCC. As a single a

{'_parse_error': 'JSONDecodeError',
 '_raw_text': '```{\n  "generic_name": "Pembrolizumab",\n  "brand_names": [\n    "Keytruda"\n  ],\n  "indications": [\n    "Melanoma",\n    "Non-Small Cell Lung Cancer (NSCLC)",\n    "Head and Neck Squamous Cell Cancer (HNSCC)",\n    "Classical Hodgkin Lymphoma (cHL)",\n    "Microsatellite Instability-High (MSI-H) or Mismatch Repair Deficient (dMMR) Cancer",\n    "Gastric Cancer",\n    "Esophageal Cancer",\n    "Cervical Cancer",\n    "Hepatocellular Carcinoma (HCC)",\n    "Merkel Cell Carcinoma (MCC)",\n    "Renal Cell Carcinoma (RCC)",\n    "Endometrial Carcinoma",\n    "Tumor Mutational Burden-High (TMB-H) Cancer",\n    "Cutaneous Squamous Cell Carcinoma (cSCC)",\n    "Triple-Negative Breast Cancer (TNBC)"\n  ],\n  "mechanism": "Pembrolizumab is a humanized monoclonal antibody that binds to the programmed cell death-1 (PD-1) receptor and blocks its interaction with PD-L1 and PD-L2. This releases PD-1 pathway-mediated inhibition of the immune respo

In [12]:
import json
import re
from typing import Any, Dict

def _safe_parse_json(text: str) -> Dict[str, Any]:
    """
    Try to parse model output as JSON with several fallbacks:
      1) Direct json.loads(text)
      2) Strip Markdown code fences ```...``` and retry
      3) Extract the first {...} block via regex and parse that

    Raises json.JSONDecodeError if all strategies fail.
    """
    # 1) Direct attempt
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    cleaned = text.strip()

    # 2) Remove Markdown code fences (``` or ```json etc.) and retry
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        # Drop any lines that look like ``` or ```json
        lines = [ln for ln in lines if not ln.strip().startswith("```")]
        cleaned_no_fence = "\n".join(lines).strip()
        try:
            return json.loads(cleaned_no_fence)
        except json.JSONDecodeError:
            cleaned = cleaned_no_fence  # fall through to regex step with this

    # 3) Extract the first {...} block (handles cases with multiple JSON objects)
    match = re.search(r"\{[\s\S]*?\}", cleaned)
    if match:
        candidate = match.group(0)
        return json.loads(candidate)

    # If we get here, all attempts failed
    raise json.JSONDecodeError("Unable to parse JSON from model output", text, 0)


In [13]:
# 5. JSON-structured drug information extraction agent (with robust parsing)

def drug_agent_json(question: str) -> Dict[str, Any]:
    """
    JSON-structured extraction variant.

    Expected JSON fields:
      - generic_name (str)
      - brand_names (list[str])
      - indications (list[str])
      - mechanism (str)
      - dosage_and_administration (str)
      - common_adverse_events (list[str])
      - serious_adverse_events (list[str])
      - boxed_warnings (list[str])
      - references (list[str])

    The function is tolerant to common model formatting issues:
      - Markdown code fences (``` or ```json)
      - Extra explanatory text around the JSON
      - Multiple JSON objects concatenated (it will use the first one)
    """
    json_instruction = (
        "Extract detailed drug information and respond ONLY as a JSON object with the "
        "following keys: generic_name, brand_names, indications, mechanism, "
        "dosage_and_administration, common_adverse_events, serious_adverse_events, "
        "boxed_warnings, references. Do NOT include any extra text outside the JSON object."
    )

    full_query = f"{json_instruction}\n\nUser question: {question}"

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[full_query],
        config=json_agent_config,
    )

    text = response.text
    # For debugging, you can uncomment:
    # print("RAW MODEL OUTPUT:\n", text)

    try:
        return _safe_parse_json(text)
    except json.JSONDecodeError as e:
        # Fallback: expose raw text so you can inspect what the model produced
        return {
            "_parse_error": "JSONDecodeError",
            "_error_msg": str(e),
            "_raw_text": text,
        }


In [14]:
# 5A. Quick tests

print("=== Stateless drug_agent test ===")
print(drug_agent("What is Keytruda used for? List main indications and common adverse events."))

print("\n=== Conversational chat_drug_agent test ===")
reset_conversation()
print("Agent:", chat_drug_agent("Hi, I mainly treat lung cancer patients. Remember this."))
print("Agent:", chat_drug_agent("Based on my practice, what Keytruda indications are most relevant?"))
print("Agent:", chat_drug_agent("What immune-related adverse events should I watch for with Keytruda?"))

print("\n=== JSON drug_agent_json test ===")
info = drug_agent_json("Extract detailed drug information for Keytruda.")
info


=== Stateless drug_agent test ===
Keytruda (pembrolizumab) is a programmed death receptor-1 (PD-1) blocking antibody used to treat multiple cancers.

**Main Indications:**
*   **Melanoma:** For patients with unresectable or metastatic melanoma and for the adjuvant treatment of adult and pediatric patients (12 years and older).
*   **Non-Small Cell Lung Cancer (NSCLC):** For the first-line treatment of metastatic NSCLC whose tumors have high PD-L1 expression (TPS ≥50%) with no EGFR or ALK genomic tumor aberrations. Also, for metastatic NSCLC whose tumors express PD-L1 (TPS ≥1%) with disease progression on or after platinum-containing chemotherapy. It is also indicated for resectable (tumors ≥4 cm or node positive) NSCLC in combination with platinum-containing chemotherapy as neoadjuvant treatment, and then continued as a single agent as adjuvant treatment after surgery.
*   **Head and Neck Squamous Cell Carcinoma (HNSCC):** For patients with recurrent or metastatic HNSCC with disease pr

{'generic_name': 'Pembrolizumab',
 'brand_names': ['Keytruda'],
 'indications': ['Melanoma',
  'Non-Small Cell Lung Cancer (NSCLC)',
  'Head and Neck Squamous Cell Cancer (HNSCC)',
  'Classical Hodgkin Lymphoma (cHL)',
  'Primary mediastinal large B-cell lymphoma (PMBCL)',
  'Urothelial Carcinoma',
  'Microsatellite Instability-High (MSI-H) or Mismatch Repair Deficient (dMMR) Cancer',
  'Gastric Cancer',
  'Esophageal Cancer',
  'Cervical Cancer',
  'Hepatocellular Carcinoma (HCC)',
  'Merkel Cell Carcinoma (MCC)',
  'Renal Cell Carcinoma (RCC)',
  'Endometrial Carcinoma',
  'Tumor Mutational Burden-High (TMB-H) Cancer',
  'Cutaneous Squamous Cell Carcinoma (cSCC)',
  'Biliary Tract Cancer (BTC)'],
 'mechanism': 'Pembrolizumab is a programmed death receptor-1 (PD-1) blocking antibody. It binds to the PD-1 receptor and blocks its interaction with PD-L1 and PD-L2, releasing PD-1 pathway-mediated inhibition of the immune response, including anti-tumor immune response.',
 'dosage_and_admin

In [15]:
# 6A. Read Excel data and convert rows into text snippets for retrieval

import pandas as pd # note, I didnt import pandas earlier

EXCEL_PATH = "data/drug_data_large.xlsx"  # Ensure this file exists in this path within this project folder

excel_df = pd.read_excel(EXCEL_PATH)

def row_to_text(row) -> str:
    """
    Convert a single Excel row into a plain-text snippet for retrieval.
    Adjust which columns are used as needed.
    """
    parts = []
    if "DrugName" in row and pd.notna(row["DrugName"]):
        parts.append(f"Drug name: {row['DrugName']}")
    if "Indication" in row and pd.notna(row["Indication"]):
        parts.append(f"Indication: {row['Indication']}")
    if "MechanismOfAction" in row and pd.notna(row["MechanismOfAction"]):
        parts.append(f"Mechanism: {row['MechanismOfAction']}")
    if "Dosage" in row and pd.notna(row["Dosage"]):
        parts.append(f"Dosage: {row['Dosage']}")
    if "Warnings" in row and pd.notna(row["Warnings"]):
        parts.append(f"Warnings: {row['Warnings']}")
    if "CommonAEs" in row and pd.notna(row["CommonAEs"]):
        parts.append(f"Common AEs: {row['CommonAEs']}")
    if "SeriousAEs" in row and pd.notna(row["SeriousAEs"]):
        parts.append(f"Serious AEs: {row['SeriousAEs']}")
    if "Notes" in row and pd.notna(row["Notes"]):
        parts.append(f"Notes: {row['Notes']}")
    return "\n".join(parts)

excel_corpus_texts = [row_to_text(r) for _, r in excel_df.iterrows()]
len(excel_corpus_texts), excel_corpus_texts[0][:200]


(15,
 'Drug name: Keytruda\nIndication: Melanoma; NSCLC; Head and Neck Cancer; MSI-H/dMMR tumors\nMechanism: PD-1 inhibitor\nDosage: 200 mg IV every 3 weeks\nWarnings: Immune-related adverse events; Pneumonitis;')

In [16]:
# 6B. Build embeddings for the Excel corpus using a Gemini embedding model

import numpy as np
from numpy.linalg import norm

EMBED_MODEL = "text-embedding-004"  # Adjust to an embedding model available to your account

def embed_text(text: str) -> np.ndarray:
    """
    Get an embedding vector for a single text using Gemini embeddings.
    """
    # Depending on the SDK version, embed_content may return embeddings on different fields
    resp = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[text],
    )
    # In current versions, this is typically resp.embeddings[0].values
    emb = resp.embeddings[0].values
    return np.array(emb, dtype="float32")

# Precompute embeddings for all Excel rows (only 15 rows, so this is fast)
excel_embeddings = np.vstack([embed_text(t) for t in excel_corpus_texts])
excel_embeddings.shape


(15, 768)

In [17]:
# 6C. Retrieval from Excel-based knowledge

def retrieve_from_excel(query: str, top_k: int = 5):
    """
    Use cosine similarity over embeddings to retrieve the top_k most relevant Excel rows.
    Returns a list of (text, score, index) tuples.
    """
    q_emb = embed_text(query)
    sims = excel_embeddings @ q_emb / (norm(excel_embeddings, axis=1) * norm(q_emb) + 1e-8)
    top_idx = np.argsort(-sims)[:top_k]

    results = []
    for i in top_idx:
        results.append((excel_corpus_texts[i], float(sims[i]), int(i)))
    return results

# Simple sanity check
test_results = retrieve_from_excel("Keytruda lung cancer indications", top_k=3)
for t, s, i in test_results:
    print("=" * 60)
    print(f"Score: {s:.3f} | Row: {i}")
    print(t)


Score: 0.745 | Row: 0
Drug name: Keytruda
Indication: Melanoma; NSCLC; Head and Neck Cancer; MSI-H/dMMR tumors
Mechanism: PD-1 inhibitor
Dosage: 200 mg IV every 3 weeks
Warnings: Immune-related adverse events; Pneumonitis; Colitis
Common AEs: Fatigue, rash, diarrhea
Serious AEs: Severe irAEs (pneumonitis, colitis, hepatitis)
Notes: Multiple tumor-agnostic approvals
Score: 0.579 | Row: 5
Drug name: Imfinzi
Indication: ES-SCLC; Biliary tract cancer; NSCLC consolidation
Mechanism: PD-L1 inhibitor
Dosage: 1500 mg IV every 3 weeks
Warnings: Immune-mediated pneumonitis
Common AEs: Fatigue, cough, nausea
Serious AEs: Severe pneumonitis
Notes: Used in combination or consolidation
Score: 0.578 | Row: 6
Drug name: Enhertu
Indication: HER2-positive metastatic breast cancer; HER2-low tumors
Mechanism: HER2-directed antibody–drug conjugate
Dosage: 5.4 mg/kg IV every 3 weeks
Warnings: Interstitial lung disease; Left ventricular dysfunction
Common AEs: Nausea, fatigue, neutropenia
Serious AEs: Severe

In [18]:
# 6D. Conversational Drug Agent WITH Excel-RAG

# Assume conversation_history / agent_config already exist above.
# For safety, initialize if not present.

try:
    conversation_history
except NameError:
    from google.genai import types  # Fallback import
    conversation_history = []

def chat_drug_agent_with_rag(user_message: str, top_k: int = 5) -> str:
    """
    Multi-turn conversational agent with Excel-based RAG.

      1) Retrieve relevant entries from the Excel knowledge base using embeddings.
      2) Build a context block from the retrieved entries.
      3) Send the context block plus the user question to the model
         (agent_config still includes google_search).
    """
    retrieved = retrieve_from_excel(user_message, top_k=top_k)
    context_texts = [t for (t, score, idx) in retrieved]

    if context_texts:
        context_block = (
            "Below is internal drug reference data from an Excel knowledge base. "
            "Use it as a primary source when relevant:\n\n"
            + "\n\n---\n\n".join(context_texts)
            + "\n\n"
        )
    else:
        context_block = "No relevant Excel rows were retrieved.\n\n"

    full_user_message = (
        f"{context_block}"
        "When answering, FIRST rely on the internal Excel data above if it covers the question. "
        "If the question cannot be answered from the Excel data, you may then use google_search. "
        "Always briefly explain which source you used.\n\n"
        f"User question: {user_message}"
    )

    # Append to history
    conversation_history.append(
        types.Content(role="user", parts=[types.Part(text=full_user_message)])
    )

    # Call the model
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=conversation_history,
        config=agent_config,  # Reuse the same config (with search_tool + system prompt)
    )

    answer = response.text

    # Append model answer to history
    conversation_history.append(
        types.Content(role="model", parts=[types.Part(text=answer)])
    )

    return answer


In [19]:
# 6E. Interactive loop using Excel-RAG-enabled agent

print("Medical Agent with Excel-RAG ready. Type 'exit' to quit this loop.")
# Reset conversation history if available
try:
    reset_conversation()
except NameError:
    conversation_history.clear()

while True:
    q = input("\nYou: ").strip()
    if q.lower() in ("exit", "quit"):
        print("Exiting Excel-RAG chat loop.")
        break

    a = chat_drug_agent_with_rag(q)
    print("Agent:", a)


Medical Agent with Excel-RAG ready. Type 'exit' to quit this loop.
Agent: Keytruda:
*   **Indication:** Melanoma, NSCLC, Head and Neck Cancer, MSI-H/dMMR tumors
*   **Mechanism:** PD-1 inhibitor
*   **Dosage:** 200 mg IV every 3 weeks
*   **Warnings:** Immune-related adverse events, Pneumonitis, Colitis
*   **Common AEs:** Fatigue, rash, diarrhea
*   **Serious AEs:** Severe irAEs (pneumonitis, colitis, hepatitis)
*   **Notes:** Multiple tumor-agnostic approvals

I used the internal Excel data to answer this question.

Exiting Excel-RAG chat loop.


In [20]:
# 7. High-level dispatcher: agent_ask

def agent_ask(q: str) -> str:
    """
    Unified entry point that routes to three underlying functions:

    - Default (no prefix):        chat_drug_agent          → conversation + search (no RAG)
    - '/rag ...' prefixed input:  chat_drug_agent_with_rag → Excel-RAG + conversation + search
    - '/json ...' prefixed input: drug_agent_json          → structured JSON extraction

    Examples:
      /rag What is the dosage of Keytruda?
      /json Extract detailed drug information for Keytruda.
    """
    text = q.strip()

    # 1) JSON extraction mode
    if text.lower().startswith("/json "):
        question = text[6:].strip()
        info = drug_agent_json(question)
        return json.dumps(info, indent=2, ensure_ascii=False)

    # 2) RAG mode (Excel + conversation)
    if text.lower().startswith("/rag "):
        question = text[5:].strip()
        return chat_drug_agent_with_rag(question)

    # 3) Default: conversational agent + search only (no Excel-RAG)
    return chat_drug_agent(text)


In [28]:
# 8. Unified interactive loop

print("Medical Agent ready. Type 'exit' to quit.")
print("Commands:")
print("  normal question → chat with search only")
print("  /rag ...        → use Excel-RAG + conversation + search")
print("  /json ...       → get structured JSON output\n")

# Use reset_conversation() if defined; otherwise clear the global history
try:
    reset_conversation()
except NameError:
    conversation_history.clear()

while True:
    q = input("You: ").strip()
    if q.lower() in ("exit", "quit"):
        print("Bye.")
        break
    if q.lower() == "/reset":
        conversation_history.clear()
        print("Agent: Memory cleared.\n")
        continue
    a = agent_ask(q)
    print("Agent:", a, "\n")


Medical Agent ready. Type 'exit' to quit.
Commands:
  normal question → chat with search only
  /rag ...        → use Excel-RAG + conversation + search
  /json ...       → get structured JSON output

Agent: Keytruda is a drug with the following characteristics:

*   **Mechanism:** PD-1 inhibitor
*   **Indications:** Melanoma, NSCLC, Head and Neck Cancer, MSI-H/dMMR tumors
*   **Dosage:** 200 mg IV every 3 weeks
*   **Warnings:** Immune-related adverse events, Pneumonitis, Colitis
*   **Common AEs:** Fatigue, rash, diarrhea
*   **Serious AEs:** Severe irAEs (pneumonitis, colitis, hepatitis)
*   **Notes:** Multiple tumor-agnostic approvals

This information is based on the internal Excel knowledge base.
 

Agent: Keytruda's sales in 2024 reached approximately $29.5 billion. This represents an 18% increase compared to the previous year.

I relied on the search results to answer this question.
 

Agent: {
  "generic_name": "Pembrolizumab",
  "brand_names": [
    "Keytruda"
  ],
  "indicati

In [ ]:
# 9. Minimal widget-based UI for Drug Information Agent
#    - Frontend has almost no logic.
#    - Raw text goes directly to `agent_ask(q)` for normal, /rag, /json.
#    - `/reset`, `exit`, `quit` behave like your original while-loop.
#    - Ctrl+Enter triggers Ask.
#    - During thinking, show "Thinking..." and then replace it with the final answer.

import ipywidgets as widgets
from IPython.display import display, Javascript

# ----------------------------------------
# Initialize backend conversation memory
# ----------------------------------------

print("Medical Agent ready. Type 'exit' to quit (console mode).")
print("Commands:")
print("  normal question → chat with search only")
print("  /rag ...        → use Excel-RAG + conversation + search")
print("  /json ...       → get structured JSON output\n")

# Same reset behavior as your loop
try:
    reset_conversation()
except NameError:
    try:
        conversation_history.clear()
    except NameError:
        pass

# ----------------------------------------
# UI Components
# ----------------------------------------

input_box = widgets.Textarea(
    value="",
    placeholder="Type your question here (supports /rag, /json, /reset, exit).",
    description="You:",
    layout={"width": "100%", "height": "80px"}
)

ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    layout={"width": "80px"}
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning",
    layout={"width": "80px"}
)

chat_output = widgets.HTML(value="")

chat_box = widgets.Box(
    [chat_output],
    layout=widgets.Layout(
        border="1px solid #cccccc",
        padding="10px",
        overflow="auto",
        height="400px",
        width="100%"
    ),
)

# Add CSS classes for JS hooks (Ctrl+Enter and auto-scroll)
chat_box.add_class("chat-scroll")
input_box.add_class("agent-input")
ask_button.add_class("agent-ask-button")

# ----------------------------------------
# UI Helper Functions
# ----------------------------------------

chat_history_html = ""


def _scroll_to_bottom():
    """Scroll the chat box to the bottom using JavaScript."""
    js_code = """
    (function() {
      const boxes = document.getElementsByClassName('chat-scroll');
      if (boxes.length > 0) {
        const box = boxes[0];
        box.scrollTop = box.scrollHeight;
      }
    })();
    """
    display(Javascript(js_code))


def add_message(role: str, text: str):
    """Render a message bubble and auto-scroll."""
    global chat_history_html

    if role == "user":
        bubble_color = "#DCF8C6"
        align = "right"
        label = "You"
    else:
        bubble_color = "#EEEEEE"
        align = "left"
        label = "Agent"

    msg_html = f"""
    <div style="width:100%; text-align:{align}; margin: 6px 0;">
        <div style="
            display:inline-block;
            padding:6px 10px;
            border-radius:10px;
            background:{bubble_color};
            max-width:75%;
            font-family:Arial, sans-serif;
            font-size:14px;
            line-height:1.4;
            white-space:pre-wrap;
        ">
            <b>{label}:</b><br>
            {text.replace('\\n','<br>')}
        </div>
    </div>
    """

    chat_history_html += msg_html
    chat_output.value = chat_history_html
    _scroll_to_bottom()


def clear_chat_ui():
    """Clear the chat area (start a brand new conversation visually)."""
    global chat_history_html
    chat_history_html = ""
    chat_output.value = ""


def show_greeting():
    """Show initial instructions for a new conversation."""
    add_message(
        "agent",
        "Medical Agent ready.\n"
        "Commands:\n"
        "  normal question → chat with search only\n"
        "  /rag ...        → use Excel-RAG + conversation + search\n"
        "  /json ...       → get structured JSON output\n"
    )


# Initial greeting for the first conversation
show_greeting()

# ----------------------------------------
# Backend interaction helper
# ----------------------------------------

def handle_query_type(raw_q: str):
    """
    Parse the input string and classify it, mirroring your while-loop:

      q = input(...).strip()
      if q.lower() in ('exit', 'quit'): ...
      if q.lower() == '/reset': ...

      else: agent_ask(q)
    """
    q = raw_q.strip()
    if not q:
        return "EMPTY", None

    ql = q.lower()
    if ql in ("exit", "quit"):
        return "EXIT", "Bye."

    if ql == "/reset":
        # Reset backend memory
        try:
            reset_conversation()
        except NameError:
            try:
                conversation_history.clear()
            except NameError:
                pass
        # UI will handle clearing the chat and re-printing greeting.
        return "RESET", None

    # Normal / RAG / JSON / anything else
    return "ASK", q


# ----------------------------------------
# Button Callbacks
# ----------------------------------------

def on_click_ask(b):
    """Handle Ask button click."""
    raw_q = input_box.value
    status, payload = handle_query_type(raw_q)

    if status == "EMPTY":
        return

    if status == "RESET":
        # Backend memory already cleared in handle_query_type.
        # Now clear UI chat and re-print greeting.
        clear_chat_ui()
        show_greeting()
        input_box.value = ""
        return

    if status == "EXIT":
        add_message("user", raw_q.strip())
        add_message("agent", payload or "Bye.")
        input_box.value = ""
        return

    # status == "ASK": normal /rag /json flows (all handled by agent_ask)
    question = payload  # already stripped
    if not question:
        return

    # Show user message
    add_message("user", question)

    # Show a temporary "Thinking..." bubble
    thinking_tag = "⏳ Thinking..."
    add_message("agent", thinking_tag)

    # Call your backend
    try:
        answer = agent_ask(question)
    except Exception as e:
        answer = f"Error calling agent_ask: {e}"

    # Replace the last "Thinking..." with the real answer.
    # Remove the last occurrence of the plain string "⏳ Thinking..."
    global chat_history_html
    chat_history_html = chat_history_html.rsplit(thinking_tag, 1)[0]
    chat_output.value = chat_history_html

    # Now append the real answer
    add_message("agent", answer)

    # Clear input box for the next question
    input_box.value = ""


def on_click_reset(b):
    """Reset button: start a new conversation (backend + UI)."""
    # Reset backend memory
    try:
        reset_conversation()
    except NameError:
        try:
            conversation_history.clear()
        except NameError:
            pass

    # Clear chat UI and input, then show greeting
    clear_chat_ui()
    show_greeting()
    input_box.value = ""


# Bind callbacks
ask_button.on_click(on_click_ask)
reset_button.on_click(on_click_reset)

# ----------------------------------------
# Assemble and Display UI
# ----------------------------------------

ui = widgets.VBox([
    widgets.HBox([ask_button, reset_button]),
    input_box,
    chat_box
])

display(ui)

# ----------------------------------------
# Add Ctrl+Enter shortcut for Ask
# ----------------------------------------

js_shortcut = """
(function() {
  // Find the widget container with class 'agent-input'
  const containers = document.getElementsByClassName('agent-input');
  if (containers.length === 0) return;
  const container = containers[0];
  const textareas = container.getElementsByTagName('textarea');
  if (textareas.length === 0) return;
  const ta = textareas[0];

  if (ta.dataset.agentBound === '1') return;
  ta.dataset.agentBound = '1';

  ta.addEventListener('keydown', function(e) {
    if (e.key === 'Enter' && e.ctrlKey) {
      e.preventDefault();
      const buttons = document.getElementsByClassName('agent-ask-button');
      if (buttons.length > 0) {
        const btnWrapper = buttons[0];
        const btnEl = btnWrapper.getElementsByTagName('button')[0] || btnWrapper;
        if (btnEl) {
          btnEl.click();
        }
      }
    }
  });
})();
"""

ask_button.add_class("agent-ask-button")
display(Javascript(js_shortcut))


Medical Agent ready. Type 'exit' to quit (console mode).
Commands:
  normal question → chat with search only
  /rag ...        → use Excel-RAG + conversation + search
  /json ...       → get structured JSON output



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>